# Barthel extraction: Gemma 3 4B, untrained vs. LoRA (Colab T4)

Adapted from Unsloth's `Gemma3_(4B).ipynb` (LGPL-3.0): same install, model and LoRA setup; data, prompt,
training settings and evaluation are ours.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. In the Colab file panel, next to this notebook: `prompt.py`, `eval.py`, `data/generate_synthetic_data.py`,
   `data/barthel_train_source/train.json`, `data/test/test.json` (the same layout as the project).
3. Run the cells in order. The **test** cells are marked: run each of them once. Hyperparameters are fixed before any test score exists.

Outputs land in `runs/` (predictions + `.meta.json` with latency and assumed cost). Download `runs_out.zip` at the end and unzip into the project's `runs/`.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 1. Check the uploaded files

In [ ]:
import os
need = ["prompt.py", "eval.py", "data/generate_synthetic_data.py",
        "data/barthel_train_source/train.json", "data/test/test.json"]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f"missing (relative to {os.getcwd()}): {missing}"
print("ok, working dir:", os.getcwd())

## 2. Shared helpers (same prompt, splits and parser as the API runs)

In [ ]:
import gc, json, os, sys, time
import torch
sys.path.insert(0, os.getcwd())
import prompt as P
import eval as ev

MAX_SEQ = 4096
GPU_USD_PER_HOUR = 0.50   # ASSUMPTION for the cost column: rough cloud price of a T4. Colab free is not billed.
MODEL_NAME = "unsloth/gemma-3-4b-it"
os.makedirs("runs", exist_ok=True)

def to_messages(note, answer=None):
    # Gemma 3 has no system role: system prompt + note go into one user turn (identical in training and inference).
    user = P.system_prompt() + "\n\n--- Diktat ---\n" + note
    msgs = [{"role": "user", "content": [{"type": "text", "text": user}]}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

def load_model():
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True, load_in_8bit=False, full_finetuning=False)
    return model, get_chat_template(tokenizer, chat_template="gemma-3")

def generate(model, tokenizer, note):
    inputs = tokenizer.apply_chat_template(
        to_messages(note), add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True).to("cuda")
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False)   # greedy: deterministic
    torch.cuda.synchronize()
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True), time.time() - t0

def run_split(name, split, model, tokenizer):
    notes = P.load_split(split)
    raw, preds, secs = {}, {}, []
    for n in notes:
        text, s = generate(model, tokenizer, n["note"])
        raw[n["note_id"]], preds[n["note_id"]] = text, P.parse_output(text)
        secs.append(s)
    stem = f"runs/{name}_{split}"
    json.dump(preds, open(stem + ".json", "w"), ensure_ascii=False, indent=1)
    json.dump(raw, open(stem + ".raw.json", "w"), ensure_ascii=False, indent=1)   # unparsed model text, for debugging
    mean = sum(secs) / len(secs)
    json.dump({"model": name, "split": split, "n_notes": len(notes), "mean_seconds_per_note": round(mean, 2),
               "usd_per_1000_notes": round(mean * 1000 / 3600 * GPU_USD_PER_HOUR, 3),
               "assumed_gpu_usd_per_hour": GPU_USD_PER_HOUR}, open(stem + ".meta.json", "w"), indent=1)
    m = ev.evaluate(notes, preds)
    print(ev.format_row(f"{name} {split}", m, ev.bootstrap(notes, preds)), f"| {mean:.1f}s/note")
    return preds

## 3. Untrained base model (val + test)
No tuning happens here, so running test is fine. This is the README's "small model, untrained" row.

In [ ]:
from unsloth import FastModel
model, tokenizer = load_model()
FastModel.for_inference(model)
run_split("gemma3-4b-base", "val", model, tokenizer)
run_split("gemma3-4b-base", "test", model, tokenizer)    # TEST (base model): run once

## 4. LoRA fine-tuning on the 180 training notes
Fixed settings: r=16, alpha=16, lr 2e-4, 3 epochs, effective batch 16, loss on the answer only.
Validation loss is logged once per epoch. Reload the base model first to free the inference copy.

In [ ]:
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = load_model()
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

def render(n):
    return tokenizer.apply_chat_template(
        to_messages(n["note"], P.target_json(n)), tokenize=False, add_generation_prompt=False).removeprefix("<bos>")

train_ds = Dataset.from_list([{"text": render(n)} for n in P.load_split("train")])
val_ds = Dataset.from_list([{"text": render(n)} for n in P.load_split("val")])
lengths = [len(tokenizer(text=t["text"]).input_ids) for t in train_ds]
print("train examples:", len(train_ds), "| max tokens:", max(lengths), "| limit:", MAX_SEQ)
assert max(lengths) < MAX_SEQ, "raise MAX_SEQ"

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=1, eval_strategy="epoch",
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=3407, report_to="none"))
trainer = train_on_responses_only(
    trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")

# Sanity check: only the answer should be trained on (the prompt shows as blanks).
row = trainer.train_dataset[0]
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in row["labels"]]).replace(tokenizer.pad_token, " ")[-600:])

In [ ]:
stats = trainer.train()
print(f"train time: {stats.metrics['train_runtime'] / 60:.1f} min")
model.save_pretrained("gemma3_lora"); tokenizer.save_pretrained("gemma3_lora")   # adapter only

## 5. Evaluate the LoRA model
Validation first. **Test: run the last cell once, after you are done with validation.**

In [ ]:
FastModel.for_inference(model)
run_split("gemma3-4b-lora", "val", model, tokenizer)

In [ ]:
run_split("gemma3-4b-lora", "test", model, tokenizer)    # TEST (LoRA model): run once

## 6. Save results

In [ ]:
import shutil
shutil.make_archive("runs_out", "zip", "runs")
shutil.make_archive("gemma3_lora_adapter", "zip", "gemma3_lora")
from google.colab import files
files.download("runs_out.zip"); files.download("gemma3_lora_adapter.zip")

Unzip `runs_out.zip` into the project's `runs/`, then locally:

```
uv run python eval.py --gold data/test/test.json --baselines --train data/barthel_train_source/train.json \
  --preds "Haiku 5.5, zero-shot=runs/claude-haiku-5-5_test_0shot.json" ... \
          "Gemma 3 4B, untrained=runs/gemma3-4b-base_test.json" "Gemma 3 4B + LoRA=runs/gemma3-4b-lora_test.json" \
  --md runs/results_test.md
```